# AIMO Interpretability — E2.6-B.1 Sampling Stability Forensics

This is a **CPU-only** diagnostic stage. It reuses `early_dynamics_128.parquet` from Stage B and does **not** regenerate model outputs.

The goal is to explain why Stage A (64 rows) and Stage B (128 rows) disagree about the strength of the 512-token Full-Trend signal.

The notebook runs:

1. first 64 vs second 64 nested grouped CV;
2. composition summaries;
3. repeated balanced and prevalence-preserving 64-group subsampling;
4. nested row-level OOF export;
5. group-bootstrap confidence intervals;
6. an automatic route decision:
   - `INDEPENDENT_STRATIFIED_REPLICATION_128`
   - `REVISE_TO_BY_1024_AND_EXTEND_2048_EOS`
   - `HETEROGENEOUS_DYNAMICS_DIAGNOSIS`

Recommended Kaggle setting: **Accelerator = None**, Internet ON if cloning GitHub / installing dependencies.


In [ ]:
# 0. Configuration

from pathlib import Path

REPO_URL = "https://github.com/luxury221/getting-started.git"
BRANCH = "research/temporal-uncertainty-v1"

# Leave None to auto-discover under /kaggle/input.
FEATURE_PARQUET_OVERRIDE = None

BUDGETS = [128, 256, 512, 1024]
PRIMARY_BUDGETS = [512, 1024]
HALF_SEEDS = [0, 1, 2, 3, 4]

SUBSAMPLE_GROUPS = 64
SUBSAMPLE_REPEATS = 50
BOOTSTRAP_REPEATS = 2000

WORK = Path("/kaggle/working")
REPO = WORK / "getting-started"
OUTPUT = WORK / "e2_6_b1_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("OUTPUT =", OUTPUT)


In [ ]:
# 1. Runtime diagnostics

import os, sys, platform, subprocess, shutil, zipfile

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())


In [ ]:
# 2. Install / validate CPU dependencies

packages = [
    "joblib==1.5.3",
    "pandas==3.0.3",
    "scikit-learn==1.8.0",
    "pyarrow>=17",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("Dependencies ready.")


In [ ]:
# 3. Clone latest research branch

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "--branch", BRANCH,
        REPO_URL,
        str(REPO),
    ],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Commit:", commit)

SCRIPT = (
    REPO
    / "solutions"
    / "uncertainty-profiling"
    / "scripts"
    / "run_sampling_stability_forensics.py"
)

assert SCRIPT.exists(), SCRIPT
print("Forensics script:", SCRIPT)


In [ ]:
# 4. Locate Stage-B early_dynamics_128.parquet

def locate_feature(override=None):
    if override is not None:
        p = Path(override)
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    direct = sorted(
        Path("/kaggle/input").rglob("early_dynamics_128.parquet")
    )
    if direct:
        print("Direct candidates:")
        for p in direct:
            print("-", p)
        return direct[0]

    zip_patterns = [
        "aimo_e2_6_stage_b_results.zip",
        "results*.zip",
        "*stage_b*.zip",
    ]
    zip_hits = []
    for pattern in zip_patterns:
        zip_hits.extend(Path("/kaggle/input").rglob(pattern))

    seen = set()
    unique_hits = []
    for p in sorted(zip_hits):
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            unique_hits.append(p)
    zip_hits = unique_hits

    if not zip_hits:
        raise FileNotFoundError(
            "Could not find early_dynamics_128.parquet or a Stage-B results ZIP."
        )

    extract_root = WORK / "_e26_b1_stageb_extract"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)

    for archive in zip_hits:
        try:
            with zipfile.ZipFile(archive, "r") as zf:
                zf.extractall(extract_root)
        except zipfile.BadZipFile:
            continue

        matches = sorted(
            extract_root.rglob("early_dynamics_128.parquet")
        )
        if matches:
            print("Extracted from:", archive)
            return matches[0]

    raise FileNotFoundError(
        "Stage-B ZIPs were found, but early_dynamics_128.parquet was not inside them."
    )

FEATURE_PARQUET = locate_feature(FEATURE_PARQUET_OVERRIDE)

print("FEATURE_PARQUET =", FEATURE_PARQUET)
print("Size MiB =", round(FEATURE_PARQUET.stat().st_size / 1024**2, 2))


In [ ]:
# 5. Run E2.6-B.1 forensics

cmd = [
    sys.executable,
    str(SCRIPT),
    "--feature-data-path", str(FEATURE_PARQUET),
    "--budgets", *[str(x) for x in BUDGETS],
    "--primary-budgets", *[str(x) for x in PRIMARY_BUDGETS],
    "--half-seeds", *[str(x) for x in HALF_SEEDS],
    "--subsample-groups", str(SUBSAMPLE_GROUPS),
    "--subsample-repeats", str(SUBSAMPLE_REPEATS),
    "--bootstrap-repeats", str(BOOTSTRAP_REPEATS),
    "--output-dir", str(OUTPUT),
]

print("Running:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=REPO,
    check=True,
)


In [ ]:
# 6. Inspect composition and first64 / second64 results

import pandas as pd
import json

composition = pd.read_csv(OUTPUT / "composition_summary.csv")
half = pd.read_csv(OUTPUT / "half_nested_deltas.csv")

print("=== Composition ===")
display(composition)

print("\n=== First64 vs Second64 nested deltas ===")
display(half)


In [ ]:
# 7. Inspect repeated subsampling stability

subsampling = pd.read_csv(OUTPUT / "subsampling_summary.csv")

display(subsampling)

print("\nPrimary interpretation columns:")
display(
    subsampling[
        [
            "mode",
            "budget",
            "median_full_ba",
            "median_full_delta_ba",
            "full_delta_p025",
            "full_delta_p975",
            "full_positive_rate",
            "median_slope_delta_ba",
            "slope_positive_rate",
        ]
    ]
)


In [ ]:
# 8. Plot repeated-subsampling Full-vs-E1 delta distributions

import matplotlib.pyplot as plt

dist = pd.read_csv(OUTPUT / "subsampling_distribution.csv")

for budget in PRIMARY_BUDGETS:
    block = dist[dist["budget"] == budget]
    data = [
        block.loc[block["mode"] == "balanced", "full_delta_ba"].to_numpy(),
        block.loc[block["mode"] == "prevalence", "full_delta_ba"].to_numpy(),
    ]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.boxplot(data, labels=["balanced", "prevalence"])
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_ylabel("Full 26D − E1 14D Balanced Accuracy")
    ax.set_title(f"E2.6-B.1 composition stability — {budget} tokens")
    plt.tight_layout()
    plt.show()


In [ ]:
# 9. Inspect group-bootstrap intervals

boot = pd.read_csv(OUTPUT / "bootstrap_summary.csv")

display(boot)

for budget in BUDGETS:
    block = boot[boot["budget"] == budget]
    print(f"\nBudget {budget}:")
    display(
        block[
            [
                "nested_seed",
                "full_delta_mean",
                "full_delta_p025",
                "full_delta_p975",
                "full_positive_rate",
                "slope_delta_mean",
                "slope_delta_p025",
                "slope_delta_p975",
            ]
        ]
    )


In [ ]:
# 10. Final B.1 route

with open(OUTPUT / "b1_decision.json", "r", encoding="utf-8") as f:
    decision = json.load(f)

print(json.dumps(decision, indent=2, ensure_ascii=False))

route = decision["decision"]["route"]
print("\n============================================================")
print("NEXT ROUTE:", route)
print("============================================================")


## Route interpretation

### `INDEPENDENT_STRATIFIED_REPLICATION_128`

512 remains stable across both balanced and prevalence-preserving subsamples.

Next GPU run should use a **randomized stratified group sample**, not the first 128 rows.

### `REVISE_TO_BY_1024_AND_EXTEND_2048_EOS`

512 is composition-sensitive but 1024 is stable.

Stop claiming a fixed 512 emergence point. Extend the next temporal experiment toward `2048 / EOS-or-4096`.

### `HETEROGENEOUS_DYNAMICS_DIAGNOSIS`

Neither 512 nor 1024 survives composition stress testing.

Do not move directly into expensive hidden-state experiments. Use the saved OOF predictions to look for conditional dynamics.


In [ ]:
# 11. Package outputs

archive = shutil.make_archive(
    "/kaggle/working/aimo_e2_6_b1_results",
    "zip",
    OUTPUT,
)

print("ZIP ready:")
print(archive)


# What to send back

Send:

```text
/kaggle/working/aimo_e2_6_b1_results.zip
```

The most important files are:

- `composition_summary.csv`
- `half_nested_deltas.csv`
- `subsampling_summary.csv`
- `bootstrap_summary.csv`
- `nested_oof_predictions.csv`
- `b1_decision.json`
